In [ ]:
import os
import os.path as op
import numpy as np
import pandas as pd
import nibabel as nib
import neuropythy as ny
from nilearn.surface import PolyMesh, PolyData, SurfaceImage
from nilearn.maskers import SurfaceLabelsMasker
from nilearn.connectome import ConnectivityMeasure

In [ ]:
# 53 participants available
PARTICIPANTS = [
    "HCA6002236_V3_MR", "HCA6005242_V2_MR", "HCA6007044_V3_MR", "HCA6007044_V4_MR",
    "HCA6016146_V2_MR", "HCA6016146_V3_MR", "HCA6018857_V3_MR", "HCA6031344_V3_MR",
    "HCA6031344_V4_MR", "HCA6037457_V3_MR", "HCA6054457_V3_MR", "HCA6062456_V3_MR",
    "HCA6072156_V3_MR", "HCA6072156_V4_MR", "HCA6130548_V2_MR", "HCA6131449_V2_MR",
    "HCA6166973_V3_MR", "HCA6174770_V2_MR", "HCA6183064_V2_MR", "HCA6228767_V3_MR",
    "HCA6276475_V3_MR", "HCA6281872_V3_MR", "HCA6283371_V2_MR", "HCA6283371_V3_MR",
    "HCA6290368_V3_MR", "HCA6290368_V4_MR", "HCA6291976_V2_MR", "HCA6302349_V3_MR",
    "HCA6330152_V3_MR", "HCA6374576_V3_MR", "HCA6397992_V2_MR", "HCA6397992_V3_MR",
    "HCA6427066_V3_MR", "HCA6429272_V3_MR", "HCA6429272_V4_MR", "HCA6464678_V3_MR",
    "HCA6474782_V3_MR", "HCA6542470_V3_MR", "HCA7348277_V3_MR", "HCA7350567_V2_MR",
    "HCA7434674_V2_MR", "HCA7452272_V2_MR", "HCA7452575_V3_MR", "HCA7453981_V2_MR",
    "HCA7453981_V3_MR", "HCA7467588_V3_MR", "HCA7469592_V3_MR", "HCA7497395_V2_MR",
    "HCA7497901_V3_MR", "HCA7519581_V1_MR", "HCA7530670_V3_MR", "HCA7530670_V4_MR",
    "HCA7536884_V2_MR",
]

main_dir  = "/home/jovyan"
local_dir = op.join(main_dir, "hcp_data")           # staged structural surfaces (per-subject upload)
atlas_dir = op.join(main_dir, "shared", "data", "atlases")

task     = "CARIT"
pe_dir   = "PA"
run_name = f"tfMRI_{task}_{pe_dir}"
func_filename_tpl = run_name + "_Atlas_MSMAll_hp0_clean_rclean_tclean.dtseries.nii"

output_dir = op.join(main_dir, "connectomes")
os.makedirs(output_dir, exist_ok=True)

In [ ]:
filename = "atl-MMPAll_space-fslr32k_hemi-{}_deterministic.label.gii"

glasser_gii_left  = nib.load(op.join(atlas_dir, filename.format("L")))
glasser_gii_right = nib.load(op.join(atlas_dir, filename.format("R")))

glasser_labels_left  = glasser_gii_left.darrays[0].data
glasser_labels_right = glasser_gii_right.darrays[0].data

glasser_ctable = glasser_gii_left.labeltable.labels
glasser_lut = pd.DataFrame({
    "index": [label.key for label in glasser_ctable],
    "name":  [label.label for label in glasser_ctable],
})

In [4]:
def compute_subject_connectome(participant):
    surf_dir = op.join(local_dir, participant, "MNINonLinear", "fsaverage_LR32k")
    func_dir = op.join(main_dir, "shared", "data", "HCP-Aging", participant, "MNINonLinear", "Results", run_name)
    func_path = op.join(func_dir, func_filename_tpl)

    left_surf_path  = op.join(surf_dir, f"{participant}.L.inflated_MSMAll.32k_fs_LR.surf.gii")
    right_surf_path = op.join(surf_dir, f"{participant}.R.inflated_MSMAll.32k_fs_LR.surf.gii")

    mesh = PolyMesh(left=left_surf_path, right=right_surf_path)

    labels_img = SurfaceImage(
        mesh = mesh,
        data = PolyData(left=glasser_labels_left, right=glasser_labels_right),
    )

    bold_cifti = nib.load(func_path)
    left_bold, right_bold = ny.hcp.cifti_split(bold_cifti)[:2]

    # SurfaceLabelsMasker expects (n_vertices, n_timepoints) per hemisphere
    n_left_vertices = nib.load(left_surf_path).darrays[0].data.shape[0]
    if left_bold.shape[0] != n_left_vertices and left_bold.shape[1] == n_left_vertices:
        left_bold, right_bold = left_bold.T, right_bold.T
    elif left_bold.shape[0] != n_left_vertices:
        raise ValueError(f"Unexpected shape {left_bold.shape} for {n_left_vertices} left-hemisphere vertices")

    func_img = SurfaceImage(mesh=mesh, data=PolyData(left=left_bold, right=right_bold))

    labels_masker = SurfaceLabelsMasker(
        labels_img  = labels_img,
        lut         = glasser_lut,
        standardize = "zscore_sample",
    ).fit()

    region_timeseries = labels_masker.transform(func_img)
    region_names = list(labels_masker.region_names_.values())

    connectome = ConnectivityMeasure(kind="correlation").fit_transform([region_timeseries])[0]

    return connectome, region_names

In [ ]:
all_edges = []
failed = []

for participant in PARTICIPANTS:
    print(f"{participant} ...", end=" ")
    try:
        connectome, region_names = compute_subject_connectome(participant)
    except Exception as e:
        print(f"FAILED: {e}")
        failed.append((participant, str(e)))
        continue

    # per-subject matrix
    pd.DataFrame(connectome, index=region_names, columns=region_names).to_csv(
        op.join(output_dir, f"{participant}_{run_name}_connectome.csv")
    )

    # long-format edges for the combined group table
    n = len(region_names)
    iu = np.triu_indices(n, k=1)
    for i, j in zip(*iu):
        all_edges.append({
            "participant": participant,
            "region_a": region_names[i],
            "region_b": region_names[j],
            "r": connectome[i, j],
        })

    print("done")

print(f"\n{len(PARTICIPANTS) - len(failed)} / {len(PARTICIPANTS)} subjects succeeded")
if failed:
    print("failed:", failed)

HCA6002236_V3_MR ... 

/tmp/ipykernel_54107/600142353.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  connectome = ConnectivityMeasure(kind="correlation").fit_transform([region_timeseries])[0]


done
HCA6005242_V2_MR ... 

/tmp/ipykernel_54107/600142353.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  connectome = ConnectivityMeasure(kind="correlation").fit_transform([region_timeseries])[0]


done
HCA6007044_V3_MR ... 

...

done
HCA7530670_V4_MR ... 

/tmp/ipykernel_54107/600142353.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  connectome = ConnectivityMeasure(kind="correlation").fit_transform([region_timeseries])[0]


done
HCA7536884_V2_MR ... 

/tmp/ipykernel_54107/600142353.py:37: FutureWarning: The default strategy for standardize is currently 'zscore' which incorrectly uses population std to calculate sample zscores. The new strategy 'zscore_sample' corrects this behavior by using the sample std. In release 0.14.0, the default strategy will be replaced by the new strategy, the 'zscore' option will be removed. and using standardize=True will fall back to 'zscore_sample'.To avoid this warning, please use 'zscore_sample' instead.
  connectome = ConnectivityMeasure(kind="correlation").fit_transform([region_timeseries])[0]


done

52 / 53 subjects succeeded
failed: [('HCA7467588_V3_MR', "No such file or no access: '/home/jovyan/shared/data/HCP-Aging/HCA7467588_V3_MR/MNINonLinear/Results/tfMRI_CARIT_PA/tfMRI_CARIT_PA_Atlas_MSMAll_hp0_clean_rclean_tclean.dtseries.nii'")]


In [ ]:
edges_df = pd.DataFrame(all_edges)
edges_df.to_csv(op.join(output_dir, f"group_{run_name}_connectome_edges_long.csv"), index=False)
edges_df.head()

,participant,region_a,region_b,r
0,HCA6002236_V3_MR,R_V1_ROI,R_MST_ROI,-0.089419
1,HCA6002236_V3_MR,R_V1_ROI,R_V6_ROI,0.220982
2,HCA6002236_V3_MR,R_V1_ROI,R_V2_ROI,0.552834
3,HCA6002236_V3_MR,R_V1_ROI,R_V3_ROI,0.392409
4,HCA6002236_V3_MR,R_V1_ROI,R_V4_ROI,0.522166
